# sklearn model with tensorflow keras tuner

In [ ]:
# ! python -m pip install --no-index --find-links=/kaggle/usr/lib/pip_install_permanent/my_packages -r /kaggle/usr/lib/pip_install_permanent/requirements.txt

In [11]:
import os
os.environ['PACKAGE_DIR'] = '/kaggle/usr/lib/pip_install_permanent'

In [2]:
from helper_func import *
# import helper_functions as hf
import sys
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import string
from spellchecker import SpellChecker
from textblob import TextBlob
from multiprocessing import Pool
from tqdm import tqdm
import numpy as np
import pandas as pd
# Preprocessing
from nltk.tokenize import word_tokenize, sent_tokenize
import operator
from spellchecker import SpellChecker
from tqdm import tqdm  # Import tqdm
import re
import inflect
from wordsegment import load, segment
from nltk.corpus import words
word_list = set(words.words())
from spellchecker import SpellChecker

from tqdm.contrib.concurrent import process_map  # If this import fails, you might need to update tqdm

import multiprocessing
 
# Import Packages
# import shutup; shutup.please()
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras_tuner as kt
import seaborn as sns

from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk import pos_tag, ne_chunk
from textblob import TextBlob

from textstat import flesch_reading_ease, smog_index

import spacy
from collections import Counter
from gensim import corpora, models
import pyLDAvis.gensim as gen
import pyLDAvis
import re

# Machine Learning & Data Preprocessing

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Deep Learning

from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Gensim
# from gensim.models import Word2Vec, KeyedVectors
import pandas as pd
# Progress bar
from tqdm import tqdm

# Keras Tuner
from keras_tuner.tuners import RandomSearch

# # Setting logging levels and environment variables
# tf.get_logger().setLevel(logging.ERROR)
# os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

from textstat import flesch_reading_ease

# from helper_functions import *
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import string
from spellchecker import SpellChecker
from textblob import TextBlob
from multiprocessing import Pool
from tqdm import tqdm
import numpy as np
import pandas as pd
# Preprocessing
from nltk.tokenize import word_tokenize, sent_tokenize
import operator
from spellchecker import SpellChecker
from tqdm import tqdm  # Import tqdm
import re
import inflect
from wordsegment import load, segment
from nltk.corpus import words
word_list = set(words.words())
print('Packages Instaled......')

/home/jack/envs/scoring/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-04-22 15:16:51.886557: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-04-22 15:16:52.509303: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Packages Instaled......


[nltk_data] Downloading package punkt to /home/jack/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/jack/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to /home/jack/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/jack/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [ ]:
# train = pd.read_csv('/kaggle/input/learning-agency-lab-automated-essay-scoring-2/train.csv')


In [ ]:
# import os

# glove_path = '/kaggle/input/embeddings/glove-840B-300d.txt'

# paragram_path = '/kaggle/input/embeddings/paragram-300-sl999.txt'

# wiki_news_path = '/kaggle/input/embeddings/wiki-news-1M-300d.vec'


# # Load embeddings

# embeddings = parallel_load_embeddings([glove_path, paragram_path, wiki_news_path])

# glove = embeddings["glove"]
# paragram = embeddings["paragram"]
# fasttext = embeddings["fasttext"]

In [ ]:
# train, glove, paragram, fastetxt = spellcheck_and_correct_text(train, embeddings)

In [ ]:
# train_essays, val_essays = custom_train_validation_split(train, test_size=0.25, random_state=42)

In [3]:
import pandas as pd

train_essays = pd.read_parquet('/home/jack/github/kaggle/scoring/train_(pre-split).parquet')
# val_essays = pd.read_parquet('/kaggle/input/newest-datasets-yo/validation_essays.parquet')

In [4]:
# pd.set_option('display.max_columns', None)

train_essays.head()

,essay_id,full_text,score,lowered,clean_text,misspelling_count,corrected_text,segmented_text,dense_vec_0,dense_vec_1,...,min_paragraph_length,avg_sentence_length,max_sentence_length,min_sentence_length,avg_word_length,max_word_length,min_word_length,flesch_reading_ease,gunning_fog_index,sentiment_score
6927,67436ca,"In our modern age, cars have become one of the...",2,"in our modern age, cars have become one of the...",in our modern age cars have become one of the ...,4,in our modern age cars have become one of the ...,in our modern age cars have become one of the ...,-0.041480,0.162677,...,176,176.0,176,176,4.886364,14,1,-98.71,73.81,-0.7227
6928,6744a28,"""Imagine a computer that knows when you're hap...",4,"""imagine a computer that knows when you're hap...",imagine a computer that knows when you are hap...,3,imagine a computer that knows when you are hap...,imagine a computer that knows when you are hap...,0.031722,0.153669,...,370,370.0,370,370,4.613514,12,1,-295.62,150.16,0.9941
6929,6745144,I agree with the use of FACS in classrooms thi...,4,i agree with the use of facs in classrooms thi...,i agree with the use of facs in classrooms thi...,11,i agree with the use of fac s in classrooms th...,i agree with the use of fac s in classrooms th...,0.027120,0.071673,...,389,389.0,389,389,4.300771,13,1,-306.45,158.07,0.9866
6930,6745748,Automated cars are the future of driving.Drive...,2,automated cars are the future of driving.drive...,automated cars are the future of driving drive...,10,automated cars are the future of driving drive...,automated cars are the future of driving drive...,-0.059321,0.123377,...,291,291.0,291,291,4.302405,13,1,-198.52,117.91,-0.3740
6931,6747056,If people participated on joining the Seagoing...,2,if people participated on joining the seagoing...,if people participated on joining the seagoing...,4,if people participated on joining the seagoing...,if people participated on joining the seagoing...,0.008032,0.129010,...,271,271.0,271,271,4.199262,12,1,-169.76,108.84,0.7428


In [5]:
train_essays['score'].value_counts()

score
3    6280
2    4723
4    3926
1    1252
5     970
6     156
Name: count, dtype: int64

In [6]:
# import pandas as pd
# from sklearn.utils import resample

# def sample_df(df, col='score', sub=0, random_state=None):
#     """
#     Balances the classes in a DataFrame by resampling.
    
#     Parameters:
#     - df: DataFrame to be resampled.
#     - col: The column name in df that contains class labels.
#     - random_state: The random state for reproducibility.
    
#     Returns:
#     - balanced_df: A DataFrame with balanced classes.
#     """
#     class_counts = df[col].value_counts()
#     target_count = max(int(class_counts.median()) - sub, class_counts.min())  # Ensure target_count is positive
    
#     balanced_df = pd.DataFrame()

#     for class_label in df[col].unique():
#         class_subset = df[df[col] == class_label]
        
#         if len(class_subset) > target_count:
#             # Downsample majority classes
#             class_subset = resample(class_subset,
#                                     replace=False,
#                                     n_samples=target_count,
#                                     random_state=random_state)
#         else:
#             # Upsample minority classes
#             class_subset = resample(class_subset,
#                                     replace=True,
#                                     n_samples=target_count,
#                                     random_state=random_state)
        
#         balanced_df = pd.concat([balanced_df, class_subset], axis=0)
    
#     # Shuffle the DataFrame to mix the classes well
#     balanced_df = balanced_df.sample(frac=1, random_state=random_state).reset_index(drop=True)

#     return balanced_df


In [7]:
# import pandas as pd
# import numpy as np


# def stratified_sample(df, target, n_samples=None, frac=None):
#     """
#     Returns a stratified sample of a dataframe based on a target column.

#     Args:
#     df (pd.DataFrame): DataFrame to sample from.
#     target (str): Name of the target column based on which stratification is done.
#     n_samples (int, optional): Total number of samples to return. If None, frac must be provided.
#     frac (float, optional): Fraction of the total per stratum to return. If None, n_samples must be provided.

#     Returns:
#     pd.DataFrame: Stratified sample of the original DataFrame.
#     """
#     # Validate inputs
#     if n_samples is None and frac is None:
#         raise ValueError("Either n_samples or frac must be provided.")
#     if n_samples is not None and frac is not None:
#         raise ValueError("Only one of n_samples or frac should be provided.")

#     # Group by the target feature and sample within groups
#     grouped = df.groupby(target)
#     if frac is not None:
#         return grouped.apply(lambda x: x.sample(frac=frac, random_state=1)).reset_index(drop=True)
#     else:
#         # Calculate fraction to meet n_samples
#         frac = n_samples / len(df)
#         return grouped.apply(lambda x: x.sample(frac=frac, random_state=1)).reset_index(drop=True)

# # Example usage



# UPSAMPLE = True

# if UPSAMPLE:
    
#     sampled_df = stratified_sample(sampled_train_essays, 'score', frac=0.4)

    
# else:
    
#     sampled_df = stratified_sample(train_essays, 'score', frac=0.4)

# print(sampled_df['score'].value_counts(normalize=True))


In [8]:
# print(train_essays['score'].value_counts(normalize=True))


In [9]:
# print(len(train_essays))
# print(len(sampled_df))

In [10]:
# SAMPLE = False

# if SAMPLE:
#     train_essays = sampled_df.copy()
    
# else:
#     train_essays = sampled_train_essays.copy()    # train_essays.copy()

In [11]:
# print(len(train_essays))
# print(len(sampled_df))
# print(len(validation_essays))

In [12]:
drop_cols = [ 'full_text', 'lowered', 'clean_text','corrected_text', 'segmented_text']


train_df = train_essays.copy()
# val_df = val_essays.copy()

train_df.drop(columns=drop_cols, inplace= True)
# val_df.drop(columns=drop_cols, inplace= True)


In [13]:
feature_cols = []

for col in train_df.columns:
    if (col != 'essay_id') and (col != 'score'):
        feature_cols.append(col) 

In [14]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import pickle
import numpy as np

# Extract labels from the training and validation datasets
train_labels = train_df['score']  # Extracts 'score' column as the label
# val_labels = val_df['score']

# Extract features for scaling
train_features = train_df[feature_cols]  # Subset with only feature columns
# val_features = val_df[feature_cols]

# Initialize the scaler
scaler = StandardScaler()  # Using StandardScaler for scaling

# Fit the scaler to the training features
scaler.fit(train_features)  # This defines the transformation based on the training data

# Transform training and validation features
train_feats_scaled = scaler.transform(train_features)  # Transforms the training data
# val_feats_scaled = scaler.transform(val_features)  # Transforms the validation data

# Reassign the scaled features to the original DataFrames, keeping the same column names
train_df[feature_cols] = train_feats_scaled  # Replace the original features with scaled ones
# val_df[feature_cols] = val_feats_scaled

# Save the scaler for later use
with open('sklearn_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)  # Persist the scaler for future use or reference


In [15]:
# Check if the file has been written correctly and is not empty
import os
scaler_path = 'sklearn_scaler.pkl'
if os.path.getsize(scaler_path) > 0:
    print(f"Scaler saved successfully in {scaler_path}.")
else:
    print(f"Failed to save scaler to {scaler_path}. File is empty.")

Scaler saved successfully in sklearn_scaler.pkl.


In [16]:
# Check if the scaler is StandardScaler
if isinstance(scaler, StandardScaler):
    print("The scaler is a StandardScaler.")
elif isinstance(scaler, MinMaxScaler):
    print("The scaler is a MinMaxScaler.")
else:
    print("The scaler is neither StandardScaler nor MinMaxScaler.")

The scaler is a StandardScaler.


In [17]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity


def pca_dataframe(df):
    """
    Apply KMeans clustering, PCA, and cosine similarity to an input DataFrame.

    Parameters:
    -----------
    df : pandas.DataFrame
        The input DataFrame containing the data to process.

    Returns:
    --------
    pandas.DataFrame, pandas.DataFrame
        The first DataFrame contains the processed data with added columns for PCA and clustering.
        The second DataFrame is the cosine similarity matrix between essays.
    """
    
    # Ensure the "essay_id" column is of type string to avoid merging issues
    df["essay_id"] = df["essay_id"].astype(str)

    # Identify dense vector columns based on a specific pattern (e.g., "vec_")
    dense_vector_columns = [col for col in df.columns if "vec_" in col]
    
    # Apply KMeans clustering with a specified number of clusters (3 in this case)
    kmeans = KMeans(n_clusters=3, random_state=42)
    cluster_labels = kmeans.fit_predict(df[dense_vector_columns])
    df["cluster"] = cluster_labels  # Add cluster labels to the DataFrame
    
    # Apply PCA to reduce dimensions, keeping 3 principal components
    pca = PCA(n_components=3)
    principal_components = pca.fit_transform(df[dense_vector_columns])
    df["PCA_1"] = principal_components[:, 0]
    df["PCA_2"] = principal_components[:, 1]
    df["PCA_3"] = principal_components[:, 2]
    
    # Compute cosine similarity across the dense vector columns
    cosine_similarity_matrix = cosine_similarity(df[dense_vector_columns])
    
    # Create a DataFrame for cosine similarity with essay IDs as index and columns
    similarity_df = pd.DataFrame(
        cosine_similarity_matrix,
        index=df["essay_id"],
        columns=df["essay_id"]
    )
    
    # Add the average cosine similarity as a new column
#     df["cosine_similarity_avg"] = similarity_df.mean(axis=1)

    # Merge the original DataFrame with the cosine similarity DataFrame on "essay_id"
    # Use 'left' join to keep the structure of the original DataFrame
    df = df.merge(similarity_df.add_prefix("cos_sim_"), left_on="essay_id", right_index=True, how="left")

    return df, similarity_df


train_df, similarity_train = pca_dataframe(train_df)
# val_df, similarity_val = pca_dataframe(val_df)

In [18]:
train_df, val_df = custom_train_validation_split(train_df, test_size=0.25, random_state=42)

In [21]:
import pandas as pd
from sklearn.utils import resample
from imblearn.over_sampling import SMOTE

df = train_df.copy()
# Downsample majority classes to 1,000 samples
target_samples = 1000
balanced_df = pd.DataFrame()

for class_label in df['score'].unique():
    class_subset = df[df['score'] == class_label]
    
    if len(class_subset) > target_samples:
        # Downsample to 1,000 samples for majority classes
        class_subset = resample(
            class_subset,
            replace=False,
            n_samples=target_samples,
            random_state=42
        )
    
    balanced_df = pd.concat([balanced_df, class_subset], axis=0)

# Now, let's apply SMOTE to balance the minority classes up to 1,000 samples
X = balanced_df.drop(columns = ['essay_id','score'], axis=1)
y = balanced_df['score']

# Create SMOTE instance to ensure all classes have 1,000 samples
smote = SMOTE(sampling_strategy={k: 1000 for k in y.unique()}, random_state=42)

X_resampled, y_resampled = smote.fit_resample(X, y)

# Create the resampled DataFrame
df_resampled = pd.DataFrame(X_resampled, columns=X.columns)
df_resampled['score'] = y_resampled

# Display the resampled class distribution to confirm the balancing
print("Resampled class distribution:")
print(df_resampled['score'].value_counts())


Resampled class distribution:
score
4    1000
2    1000
3    1000
5    1000
1    1000
6    1000
Name: count, dtype: int64


In [22]:
UPSAMPLE = True

if UPSAMPLE:
    
    # Balance only the training DataFrame
    train_df = df_resampled.copy()

    
else:
    pass

print(train_df['score'].value_counts())
print(val_df['score'].value_counts())

score
4    1000
2    1000
3    1000
5    1000
1    1000
6    1000
Name: count, dtype: int64
score
3    1590
2    1176
4     987
1     295
5     242
6      37
Name: count, dtype: int64


In [23]:
# To convert features and targets to NumPy arrays for ML use
train_features = train_df.drop(columns=['score'], axis=1).values
train_labels = train_df['score'].values

val_features = val_df.drop(columns=['essay_id','score'], axis=1).values
val_labels = val_df['score'].values


print("Features shape:", train_features.shape)
print("Target shape:", train_labels.shape)

print("Features shape:", val_features.shape)
print("Target shape:", val_labels.shape)

Features shape: (6000, 17627)
Target shape: (6000,)
Features shape: (4327, 17627)
Target shape: (4327,)


In [24]:
# import keras_tuner
# from sklearn import ensemble
# from sklearn import linear_model
# from sklearn import model_selection
# from sklearn import metrics
# from sklearn.metrics import make_scorer, cohen_kappa_score

# def build_model(hp):
#     """
#     Builds a machine learning model based on hyperparameters.
    
#     Parameters:
#     hp : HyperParameters
#         Hyperparameters for tuning the model.
    
#     Returns:
#     model : An instance of a Scikit-learn model.
#     """
#     model_type = hp.Choice('model_type', ['random_forest', 'ridge'])
#     if model_type == 'random_forest':
#         model = ensemble.RandomForestClassifier(
#             n_estimators=hp.Int('n_estimators', 10, 50, step=10),
#             max_depth=hp.Int('max_depth', 3, 10),
#             min_samples_split=hp.Int('min_samples_split', 2, 10),
#             min_samples_leaf=hp.Int('min_samples_leaf', 1, 10),
#             criterion=hp.Choice('criterion', ['gini', 'entropy']),
#             class_weight=hp.Choice('class_weight', ['balanced', 'balanced_subsample']),
#             max_samples=hp.Float('max_samples', 0.1, 1.0, sampling='log'))
#     else:
#         model = linear_model.RidgeClassifier(
#             alpha=hp.Float('alpha', 1e-3, 1, sampling='log'))

#     return model

def quadratic_weighted_kappa_scorer(y_true, y_pred):
    """
    Compute the Quadratic Weighted Kappa (QWK), also known as Cohen's kappa.
    
    Parameters:
    y_true : array-like of shape (n_samples,)
        True labels.
    y_pred : array-liimport keras_tuner
from sklearn import ensemble
from sklearn import datasets
from sklearn import linear_model
from sklearn import metrics
from sklearn import model_selection

def build_model(hp):
  model_type = hp.Choice('model_type', ['random_forest', 'ridge'])
  if model_type == 'random_forest':
    model = ensemble.RandomForestClassifier(
        n_estimators=hp.Int('n_estimators', 10, 50, step=10),
        max_depth=hp.Int('max_depth', 3, 10))
  else:
    model = linear_model.RidgeClassifier(
        alpha=hp.Float('alpha', 1e-3, 1, sampling='log'))
  return model

tuner = keras_tuner.tuners.SklearnTuner(
    oracle=keras_tuner.oracles.BayesianOptimizationOracle(
        objective=keras_tuner.Objective('score', 'max'),
        max_trials=100),
    hypermodel=build_model,
    scoring=metrics.make_scorer(metrics.accuracy_score),
    cv=model_selection.StratifiedKFold(10),
    directory='.',
    project_name='my_project')ke of shape (n_samples,)
        Predicted labels.
    
    Returns:
    score : float
        Quadratic Weighted Kappa score.
    """
    return cohen_kappa_score(y_true, y_pred, weights='quadratic')


# # Wrapping QWK as a custom scorer for model evaluation
# qwk_scorer = make_scorer(quadratic_weighted_kappa_scorer)

# # Tuner configuration
# tuner = keras_tuner.tuners.SklearnTuner(
#     oracle=keras_tuner.oracles.BayesianOptimizationOracle(
#         objective=keras_tuner.Objective('score', 'max'),
#         max_trials=100),
#     hypermodel=build_model,
#     scoring=qwk_scorer,  # Use the QWK scorer
#     cv=model_selection.StratifiedKFold(10),
#     directory='.',
#     project_name='data/sklearn',
#     overwrite=True)


# tuner.search(train_features, train_labels)

# best_model = tuner.get_best_models(num_models=1)[0]

In [25]:
# from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
# import keras_tuner as kt

# model_checkpoint = ModelCheckpoint('data/models/best_standard_model_epoch.keras', 
#                                    save_best_only=True, monitor='val_loss', mode='min')

# early_stopping = EarlyStopping(monitor='val_loss', patience=5, 
#                                restore_best_weights=True)

# call_backs = [model_checkpoint, early_stopping]

In [26]:
import keras_tuner
from sklearn import ensemble, linear_model, model_selection, svm
from sklearn.metrics import make_scorer, cohen_kappa_score
from lightgbm import LGBMClassifier


def build_model(hp):
    """
    Builds a more comprehensive machine learning model based on hyperparameters for multiclass classification.
    
    Parameters:
    hp : HyperParameters
        Hyperparameters for tuning the model.
    
    Returns:
    model : An instance of a Scikit-learn model.
    """
    # Adding 'lightgbm' as a new model type
    model_type = hp.Choice('model_type', ['random_forest',  'gradient_boosting', 'lightgbm'])

    if model_type == 'random_forest':
        model = ensemble.RandomForestClassifier(
            n_estimators=hp.Int('n_estimators', 10, 100, step=10),
            max_depth=hp.Int('max_depth', 3, 20),
            min_samples_split=hp.Int('min_samples_split', 2, 20),
            min_samples_leaf=hp.Int('min_samples_leaf', 1, 10),
            criterion=hp.Choice('criterion', ['gini', 'entropy']),
            class_weight=hp.Choice('class_weight', ['balanced', 'balanced_subsample']),
            max_samples=hp.Float('max_samples', 0.1, 1.0, sampling='log')
        )



    elif model_type == 'gradient_boosting':
        model = ensemble.GradientBoostingClassifier(
            n_estimators=hp.Int('n_estimators', 50, 200, step=50),
            learning_rate=hp.Float('learning_rate', 0.01, 0.2, sampling='log'),
            max_depth=hp.Int('max_depth', 3, 15),
            subsample=hp.Float('subsample', 0.5, 1.0, step=0.1)
        )

    # Adding LightGBM as a new model type
    elif model_type == 'lightgbm':
        model = LGBMClassifier(
            n_estimators=hp.Int('n_estimators', 50, 200, step=50),
            num_leaves=hp.Int('num_leaves', 31, 127, step=16),
            learning_rate=hp.Float('learning_rate', 0.01, 0.2, sampling='log'),
            min_child_samples=hp.Int('min_child_samples', 10, 50, step=10),
            class_weight=hp.Choice('class_weight', ['balanced', None])
        )

    return model


# Defining the custom QWK scorer
qwk_scorer = make_scorer(quadratic_weighted_kappa_scorer)

# Tuner configuration
tuner = keras_tuner.tuners.SklearnTuner(
    oracle=keras_tuner.oracles.BayesianOptimizationOracle(
        objective=keras_tuner.Objective('score', 'max'),
        max_trials=12),
    hypermodel=build_model,
    scoring=qwk_scorer,
    cv=model_selection.StratifiedKFold(3),
    directory='.',
    project_name='data/sklearn',
    overwrite=True)

# Starting the search
tuner.search(train_features, train_labels)      # class_weight=class_weights_dict,

# Retrieving the best model
best_model = tuner.get_best_models(num_models=1)[0]



Search: Running Trial #1

Value             |Best Value So Far |Hyperparameter
gradient_boosting |gradient_boosting |model_type
20                |20                |n_estimators
9                 |9                 |max_depth
18                |18                |min_samples_split
4                 |4                 |min_samples_leaf
gini              |gini              |criterion
balanced_subsample|balanced_subsample|class_weight
0.61683           |0.61683           |max_samples



In [ ]:
# sampled_train_labels.unique()

In [ ]:
best_model.fit(train_features, train_labels)

In [ ]:
predictions = best_model.predict(val_features)

y_true = val_labels

# predictions = target_scaler.inverse_transform(predictions.reshape(-1, 1)).flatten()


In [ ]:
# confusion matrix

from sklearn.metrics import confusion_matrix

confusion_matrix(y_true, predictions)

In [ ]:
# classification report

from sklearn.metrics import classification_report

print(classification_report(y_true, predictions))

In [ ]:
# cohens kappa

from sklearn.metrics import cohen_kappa_score

cohen = cohen_kappa_score(y_true, predictions)

In [ ]:
# quadratic weighted kappa

from sklearn.metrics import cohen_kappa_score

quadratic = cohen_kappa_score(y_true, predictions, weights='quadratic')

print(f"Cohen's Kappa: {cohen}")
print(f"Quadratic Weighted Kappa: {quadratic}")

In [ ]:
from joblib import dump, load

dump(best_model, 'standard_random_forest.joblib') 

In [ ]:
# forest_model = load('random_forest.joblib') 

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Generate and display the confusion matrix for the test predictions
cm = confusion_matrix(y_true, predictions, labels=best_model.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=best_model.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()
